# Module 5 실습: MCP 전송 계층(Transport) 및 배포 전략

이 노트북은 **Module 5: 전송 계층(Transport) 및 배포 전략** 실습용입니다.

학습 목표:
- **STDIO** 전송 방식의 의미와 사용법 이해
- **Streamable HTTP** 전송 방식의 의미와 사용법 이해
- 왜 **로컬 단일 사용자**는 STDIO가, **원격 멀티 클라이언트**는 HTTP가 적합한지 이해
- **VS Code** 와 **Claude Desktop** 에 연결할 때 어떤 설정이 필요한지 정리
- Colab 안에서 **실제로 STDIO와 HTTP 둘 다 테스트**


## 실습 실행 순서

아래 순서대로 실행하세요.

1. **FastMCP 설치**
2. **기본 import**
3. **데모 MCP 서버 파일 생성**
4. **STDIO transport로 서버에 연결**
5. **STDIO로 tool / resource / prompt 확인**
6. **HTTP transport로 서버 실행**
7. **Streamable HTTP로 서버에 연결**
8. **HTTP로 tool / resource / prompt 확인**
9. **SSE(legacy) 실행 예시 확인**
10. **VS Code / Claude Desktop 설정 예시 파일 생성**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


## 핵심 개념 요약

- **STDIO**: 클라이언트가 서버 프로세스를 직접 실행하고, `stdin/stdout` 으로 JSON-RPC 메시지를 주고받는 방식
- **Streamable HTTP**: 서버가 독립 프로세스로 실행되고, 여러 클라이언트가 HTTP로 접속하는 방식
- **SSE**: 과거 HTTP+SSE 방식과의 호환을 위한 legacy 옵션
- **로컬 IDE/데스크톱 앱**: 보통 STDIO가 가장 단순
- **클라우드 배포 / 멀티 클라이언트**: 보통 Streamable HTTP가 적합


In [ ]:
# 1) 설치
!pip -q install -U fastmcp


In [ ]:
# 2) 기본 import
import os
import sys
import json
import time
import socket
import signal
import subprocess
from pathlib import Path

from fastmcp import Client
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport

print("import 완료")


## 1) 데모 MCP 서버 파일 생성

이 서버는 다음을 제공합니다.

- Tool: `add_numbers`
- Resource: `resource://module5/info`
- Prompt: `explain_transport`

서버 파일은 환경 변수에 따라 transport를 바꿔 실행할 수 있게 만듭니다.

- 기본값: `stdio`
- `MCP_TRANSPORT=http` 이면 HTTP 서버로 실행
- `MCP_TRANSPORT=sse` 이면 legacy SSE 서버로 실행


In [ ]:
SERVER_CODE = r"""
from fastmcp import FastMCP
from fastmcp.prompts import Message
import os

mcp = FastMCP("Module5TransportDemo")

@mcp.tool
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더합니다."""
    return a + b

@mcp.resource(
    "resource://module5/info",
    name="module5_info",
    description="Module 5 transport 실습 안내",
    mime_type="text/plain"
)
def module5_info() -> str:
    return (
        "Module 5 실습 서버입니다. "
        "STDIO는 로컬 연동에, HTTP는 원격/멀티 클라이언트에 적합합니다."
    )

@mcp.prompt
def explain_transport(transport_name: str, audience: str = "beginner") -> list[Message]:
    """전송 방식을 설명하는 재사용 가능한 프롬프트를 생성합니다."""
    return [
        Message(
            f"You are an instructor. Explain the MCP transport '{transport_name}' "
            f"for a {audience} audience. Keep the explanation concise."
        )
    ]

if __name__ == "__main__":
    transport = os.getenv("MCP_TRANSPORT", "stdio")
    host = os.getenv("MCP_HOST", "127.0.0.1")
    port = int(os.getenv("MCP_PORT", "8001"))
    path = os.getenv("MCP_PATH", "/mcp")

    if transport in ("http", "streamable-http"):
        mcp.run(transport="http", host=host, port=port, path=path)
    elif transport == "sse":
        mcp.run(transport="sse", host=host, port=port)
    else:
        mcp.run()
"""

server_path = Path("module5_demo_server.py")
server_path.write_text(SERVER_CODE, encoding="utf-8")

print(f"서버 파일 생성 완료: {server_path.resolve()}")


## 2) STDIO transport 실습

STDIO에서는 **클라이언트가 서버 프로세스를 직접 실행**합니다.  
즉, 서버를 따로 먼저 띄우는 것이 아니라, `StdioTransport(...)` 가 내부적으로 subprocess를 시작합니다.


In [ ]:
def resource_items_to_text(items):
    out = []
    for item in items:
        if hasattr(item, "text"):
            out.append({"mimeType": getattr(item, "mimeType", None), "text": item.text})
        else:
            out.append(str(item))
    return out

def prompt_messages_to_dict(prompt_result):
    result = []
    for msg in prompt_result.messages:
        role = getattr(msg, "role", None)
        content = getattr(msg, "content", None)
        text = getattr(content, "text", str(content))
        result.append({"role": role, "content": text})
    return result

print("헬퍼 함수 준비 완료")


In [ ]:
# STDIO transport 생성
stdio_transport = StdioTransport(
    command=sys.executable,
    args=[str(server_path)],
    env={},           # 필요한 환경변수가 있다면 여기서 명시적으로 전달
    keep_alive=False  # 실습용: 연결 종료 시 서버도 같이 종료
)

print("STDIO transport 준비 완료")


In [ ]:
async def demo_stdio():
    client = Client(stdio_transport)

    async with client:
        print("=== STDIO: list_tools ===")
        tools = await client.list_tools()
        for t in tools:
            name = getattr(t, "name", None)
            description = getattr(t, "description", None)
            print({"name": name, "description": description})

        print("\n=== STDIO: call_tool(add_numbers) ===")
        tool_result = await client.call_tool("add_numbers", {"a": 10, "b": 20})
        print("structured data:", getattr(tool_result, "data", None))
        print("content:", getattr(tool_result, "content", None))

        print("\n=== STDIO: read_resource(resource://module5/info) ===")
        resource_result = await client.read_resource("resource://module5/info")
        print(json.dumps(resource_items_to_text(resource_result), ensure_ascii=False, indent=2))

        print("\n=== STDIO: get_prompt(explain_transport) ===")
        prompt_result = await client.get_prompt(
            "explain_transport",
            {"transport_name": "stdio", "audience": "beginner"}
        )
        print(json.dumps(prompt_messages_to_dict(prompt_result), ensure_ascii=False, indent=2))

await demo_stdio()


## 3) HTTP transport 실습

이번에는 서버를 **독립 프로세스**로 띄운 뒤,  
클라이언트가 URL로 접속하는 **Streamable HTTP** 방식으로 테스트합니다.


In [ ]:
HTTP_PORT = 8001
HTTP_PATH = "/mcp"
HTTP_URL = f"http://127.0.0.1:{HTTP_PORT}{HTTP_PATH}"

http_env = os.environ.copy()
http_env["MCP_TRANSPORT"] = "http"
http_env["MCP_HOST"] = "127.0.0.1"
http_env["MCP_PORT"] = str(HTTP_PORT)
http_env["MCP_PATH"] = HTTP_PATH

http_proc = subprocess.Popen(
    [sys.executable, str(server_path)],
    env=http_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"HTTP 서버 프로세스 시작 pid={http_proc.pid}")


In [ ]:
def wait_for_port(host: str, port: int, timeout: float = 15.0):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.2)
    return False

ready = wait_for_port("127.0.0.1", HTTP_PORT, timeout=20.0)
print("HTTP 서버 준비 상태:", ready)

if not ready:
    if http_proc.poll() is not None:
        print("서버가 조기 종료되었습니다. 출력 로그:")
        print(http_proc.stdout.read())
    raise RuntimeError("HTTP 서버가 제시간에 시작되지 않았습니다.")


In [ ]:
http_transport = StreamableHttpTransport(url=HTTP_URL)
print("HTTP transport 준비 완료:", HTTP_URL)


In [ ]:
async def demo_http():
    client = Client(http_transport)

    async with client:
        print("=== HTTP: list_tools ===")
        tools = await client.list_tools()
        for t in tools:
            name = getattr(t, "name", None)
            description = getattr(t, "description", None)
            print({"name": name, "description": description})

        print("\n=== HTTP: call_tool(add_numbers) ===")
        tool_result = await client.call_tool("add_numbers", {"a": 7, "b": 8})
        print("structured data:", getattr(tool_result, "data", None))
        print("content:", getattr(tool_result, "content", None))

        print("\n=== HTTP: read_resource(resource://module5/info) ===")
        resource_result = await client.read_resource("resource://module5/info")
        print(json.dumps(resource_items_to_text(resource_result), ensure_ascii=False, indent=2))

        print("\n=== HTTP: get_prompt(explain_transport) ===")
        prompt_result = await client.get_prompt(
            "explain_transport",
            {"transport_name": "streamable-http", "audience": "manager"}
        )
        print(json.dumps(prompt_messages_to_dict(prompt_result), ensure_ascii=False, indent=2))

await demo_http()


In [ ]:
# HTTP 서버 종료
if 'http_proc' in globals() and http_proc.poll() is None:
    http_proc.terminate()
    try:
        http_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        http_proc.kill()
        http_proc.wait(timeout=5)

print("HTTP 서버 종료 완료")


## 4) SSE(legacy) 실행 예시

MCP 최신 사양에서는 **HTTP+SSE 대신 Streamable HTTP가 표준**입니다.  
다만 FastMCP는 구형 클라이언트 호환을 위해 `sse` transport도 제공합니다.

아래 셀은 **실행 예시 명령만 생성**합니다.


In [ ]:
SSE_PORT = 8002
sse_command = {
    "env": {
        "MCP_TRANSPORT": "sse",
        "MCP_HOST": "127.0.0.1",
        "MCP_PORT": str(SSE_PORT),
    },
    "command": f"{sys.executable} {server_path}"
}

print("SSE legacy 실행 예시:")
print(json.dumps(sse_command, ensure_ascii=False, indent=2))
print(f"예상 SSE 엔드포인트 예시는 보통 http://127.0.0.1:{SSE_PORT}/sse 형태입니다.")


## 5) VS Code 설정 예시 파일 생성

VS Code는 `mcp.json` 파일로 MCP 서버를 설정합니다.

이 예제에서는:
- **stdio 로컬 서버**
- **http 원격/로컬 URL 서버**

두 가지 예시를 만듭니다.


In [ ]:
vscode_config = {
    "servers": {
        "module5-demo-stdio": {
            "type": "stdio",
            "command": sys.executable,
            "args": [str(server_path.resolve())]
        },
        "module5-demo-http": {
            "type": "http",
            "url": HTTP_URL
        }
    }
}

vscode_config_path = Path("vscode_mcp_example.json")
vscode_config_path.write_text(json.dumps(vscode_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("VS Code용 설정 파일 생성 완료:")
print(vscode_config_path.resolve())
print(json.dumps(vscode_config, ensure_ascii=False, indent=2))


## 6) Claude Desktop 설정 예시 생성

현재 Claude Desktop에서는:
- **로컬 MCP 서버**: 데스크톱 확장(Desktop Extension) 방식이 더 쉬운 흐름
- **원격 MCP 서버**: **Settings > Connectors** 에서 추가하는 방식이 공식 안내

다만, 로컬 stdio 서버를 위한 `claude_desktop_config.json` 예시도 여전히 유용할 수 있으므로,
여기서는 **로컬 stdio 예시 JSON** 을 생성합니다.


In [ ]:
claude_desktop_local_config = {
    "mcpServers": {
        "module5-demo-local": {
            "type": "stdio",
            "command": sys.executable,
            "args": [str(server_path.resolve())],
            "env": {}
        }
    }
}

claude_config_path = Path("claude_desktop_config_example.json")
claude_config_path.write_text(
    json.dumps(claude_desktop_local_config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Claude Desktop 로컬 stdio 설정 예시 파일 생성 완료:")
print(claude_config_path.resolve())
print(json.dumps(claude_desktop_local_config, ensure_ascii=False, indent=2))


## 7) 배포 전략 정리

### 언제 STDIO를 쓰는가
- 로컬 개발
- 단일 사용자 도구
- Claude Desktop / IDE 로컬 연동
- 네트워크 노출이 필요 없는 경우

### 언제 Streamable HTTP를 쓰는가
- 원격 서버 배포
- 여러 클라이언트 동시 접속
- 중앙 집중형 MCP 서비스
- 인증, 로깅, 헬스체크, 운영 모니터링이 필요한 경우

### 실무 팁
- 새 프로젝트는 **HTTP(Streamable HTTP)** 를 우선 고려
- 로컬 데스크톱 연동은 **STDIO** 가 가장 단순
- SSE는 **레거시 호환용** 으로만 유지


## 실습 정리

이번 노트북에서 확인한 핵심:

1. **STDIO** 에서는 클라이언트가 서버 프로세스를 직접 띄운다.
2. **HTTP** 에서는 서버가 독립적으로 떠 있고, 클라이언트가 URL로 접속한다.
3. 같은 MCP 서버라도 transport만 바꿔 여러 배포 형태를 지원할 수 있다.
4. **VS Code** 는 `mcp.json` 으로 서버를 관리한다.
5. **Claude Desktop** 은 현재 로컬은 데스크톱 확장 흐름이 쉬워졌고, 원격은 Connectors에서 추가하는 방식이 공식 흐름이다.
